Teams building with multimodal data stitch together 5-8 services: a database, a vector store, blob storage, an orchestrator, a RAG framework, an experiment tracker, custom retry logic. **Pixeltable replaces them with one table where every modality is a first-class column type.**

| Instead of ... | Pixeltable gives you ... |
|---|---|
| PostgreSQL / MySQL | `pxt.create_table()` with schema in Python, versioned automatically |
| Pinecone / Weaviate / Qdrant | `add_embedding_index()` in one line, always in sync |
| S3 / boto3 / blob storage | `pxt.Image`, `Video`, `Audio`, `Document` as native types |
| Airflow / Prefect / Celery | Computed columns that trigger on insert, no orchestrator |
| LangChain / LlamaIndex | `@pxt.query` + `.similarity()` with computed column chaining |

### Built for AI-first development

Pixeltable integrates with [20+ AI providers](https://docs.pixeltable.com/integrations/frameworks) (OpenAI, Anthropic, Gemini, HuggingFace, Ollama, Bedrock, ...) and is designed to work with AI coding assistants. The codebase is fully declarative Python, so LLMs can generate correct Pixeltable code without complex glue logic.

- **[llms.txt](https://docs.pixeltable.com/llms.txt)** for full docs in LLM-readable format
- **[MCP Server](https://github.com/pixeltable/mcp-server-pixeltable-developer)** for interactive table exploration
- **[Pixeltable Starter Kit](https://github.com/pixeltable/pixeltable-starter-kit)** for a production reference app (FastAPI + React)

```bash
pip install pixeltable google-genai 'fastapi[standard]'
pip install torch transformers  # optional, for object detection
```

In [ ]:
import os, getpass

if 'GEMINI_API_KEY' not in os.environ and 'GOOGLE_API_KEY' not in os.environ:
    os.environ['GEMINI_API_KEY'] = getpass.getpass('Gemini API Key: ')

import pixeltable as pxt
from pixeltable.functions import gemini

BASE_URL = 'https://raw.githubusercontent.com/pixeltable/pixeltable/release/docs/resources'

## 1. Store: Multimodal Tables

Video, audio, images, and documents are first-class column types, not opaque blobs in S3. `pip install pixeltable` is all you need.

In [ ]:
pxt.drop_dir('demo', force=True, if_not_exists='ignore')
pxt.create_dir('demo')

videos = pxt.create_table('demo/videos', {'video': pxt.Video, 'title': pxt.String})
videos

## 2. Orchestrate: AI as Computed Columns

No orchestrator, no DAG. Add a computed column and Pixeltable calls Gemini on every insert, caches results, retries failures, and keeps embeddings in sync.

In [ ]:
videos.add_computed_column(
    response=gemini.generate_content(
        [videos.video, 'Describe this video in detail.'], model='gemini-3-flash-preview'
    )
)

videos.add_computed_column(
    description=videos.response.candidates[0].content.parts[0].text.astype(pxt.String)
)

videos.add_embedding_index('description', embedding=gemini.embed_content.using(model='gemini-embedding-2-preview'))

## 3. Insert: One Call Triggers the Full Pipeline

A single `insert()` downloads the videos, runs Gemini analysis, extracts the description text, and computes embeddings. Open the **Dashboard** to watch it happen in real time.

In [ ]:
videos.insert([
    {'video': f'{BASE_URL}/bangkok.mp4', 'title': 'Bangkok Street Tour'},
    {'video': f'{BASE_URL}/The-Pursuit-of-Happiness-Video-Extract.mp4', 'title': 'The Pursuit of Happiness'},
])

In [ ]:
videos = pxt.get_table('demo/videos')
videos.select(videos.title, videos.description).collect()

## 4. Retrieve: Semantic Search

Search by meaning, not keywords. The embedding index stays in sync automatically; no separate vector DB to manage.

In [ ]:
sim = videos.description.similarity(string='street food')
videos.order_by(sim, asc=False).limit(5).select(videos.title, sim).collect()

## 5. Compose: On-the-Fly Object Detection

Chain any model at query time. Here we extract a frame from each video and run HuggingFace DETR for object detection. Nothing is pre-computed; the model runs on the fly.

In [ ]:
from pixeltable.functions import huggingface

videos.select(
    videos.title,
    detections=huggingface.detr_for_object_detection(
        videos.video.extract_frame(timestamp=2.0), model_id='facebook/detr-resnet-50',
    ),
).collect()

## 6. Serve: Queries Become API Endpoints

Turn any `@pxt.query` into an HTTP endpoint with `FastAPIRouter`. No hand-written routes, no Pydantic models, no serialization code. In production, use `pxt serve service.toml`.

In [ ]:
import json
import fastapi
from fastapi.testclient import TestClient
from pixeltable.serving import FastAPIRouter

@pxt.query
def search_videos(query_text: str, limit: int = 5):
    sim = videos.description.similarity(string=query_text)
    return videos.order_by(sim, asc=False).limit(limit).select(videos.title, videos.description, sim)

app = fastapi.FastAPI()
router = FastAPIRouter()
router.add_query_route(path='/search', query=search_videos)
router.add_insert_route(videos, path='/ingest', inputs=['video', 'title'], outputs=['title', 'description'])
app.include_router(router)

client = TestClient(app)

In [ ]:
resp = client.post('/search', json={'query_text': 'street food', 'limit': 2})
resp.json()

In [ ]:
resp = client.post('/ingest', json={'video': f'{BASE_URL}/bangkok.mp4', 'title': 'Test Upload'})
resp.json()

## Bonus: Cloud Storage (Optional)

Every Pixeltable Cloud account includes a free managed storage bucket. Set two env vars and all computed media flows to the cloud instead of local disk. No S3 account, no bucket creation.

```toml
# ~/.pixeltable/config.toml
[pixeltable]
api_key = "your-pixeltable-cloud-api-key"
output_media_dest = "pxtfs://yourorg:yourdb/home"
```

Or configure at runtime (skip by pressing Enter if you don't have a key yet):

In [ ]:
import os

if 'PIXELTABLE_API_KEY' not in os.environ:
    key = getpass.getpass('Pixeltable Cloud API Key (Enter to skip): ')
    if key:
        os.environ['PIXELTABLE_API_KEY'] = key

if os.environ.get('PIXELTABLE_API_KEY'):
    os.environ['PIXELTABLE_OUTPUT_MEDIA_DEST'] = 'pxtfs://yourorg:yourdb/home'  # replace with your org:db
    print(f'Cloud storage enabled: {os.environ["PIXELTABLE_OUTPUT_MEDIA_DEST"]}')
else:
    print('Skipped. Using local storage.')

## What Pixeltable Does

Everything you saw in this demo is part of a broader system. Here's the full picture:

| You write | Pixeltable does |
|---|---|
| `pxt.Image`, `pxt.Video`, `pxt.Document` columns | Stores media, handles formats, caches from URLs |
| `add_computed_column(fn(...))` | Runs incrementally, caches results, retries failures |
| `add_embedding_index(column)` | Manages vector storage, keeps index in sync |
| `@pxt.udf` / `@pxt.query` | Creates reusable functions with dependency tracking |
| `table.insert(...)` | Triggers all dependent computations automatically |
| `t.sample(5).select(t.text, summary=udf(t.text))` | Experiments on a sample; nothing stored, calls parallelized and cached |
| `table.select(...).collect()` | Returns structured + unstructured data together |
| *(nothing; it's automatic)* | Versions all data and schema changes for time-travel |

### The ten primitives

| Primitive | What it gives you |
|---|---|
| **Store** | Multimodal tables with native `Image`, `Video`, `Audio`, `Document` types |
| **Orchestrate** | AI providers as computed columns; trigger on insert, no DAG |
| **Iterate** | Views with iterators for frames, chunks, segments |
| **Index** | Embedding indexes that stay in sync with the data |
| **Extend** | `@pxt.udf` for custom logic, any Python library |
| **Agents & Tools** | `@pxt.tools` + `invoke_tools()` for tool-calling agents as computed columns |
| **Query & Experiment** | `.sample()`, `.select()` with on-the-fly expressions; prototype to production |
| **Serve** | `@pxt.query` + `FastAPIRouter` or `pxt serve` for HTTP endpoints |
| **Version** | Automatic versioning, `history()`, `revert()`, time-travel |
| **Import/Export** | CSV, JSON, Parquet, PyTorch, COCO, LanceDB, SQL, Hugging Face |

### Go deeper

- **[Pixeltable Starter Kit](https://github.com/pixeltable/pixeltable-starter-kit)** -- Full-stack reference app: document/image/video processing, cross-modal search, and a tool-calling agent, all built on computed columns
- **[Documentation](https://docs.pixeltable.com/)** -- Tutorials, cookbooks, deployment guides
- **[GitHub](https://github.com/pixeltable/pixeltable)** -- Apache 2.0, free, no account required